# Stale Labels Bug: Before vs After Performance Comparison

**Context:** Models were previously trained on `data/legacy/summary_extended.csv`, which contained labels from an older synthetic data generation run. The transaction counts and risk labels in that file did not match the actual files in `data/user_transactions/`. This notebook quantifies the impact of that bug by training Logistic Regression and XGBoost on both label sets and comparing evaluation metrics.

| Label File | Source | Status |
|---|---|---|
| `data/legacy/summary_extended.csv` | Old generation run | **STALE (bug)** |
| `data/user_labels.csv` | Current generation run | **CORRECT (fixed)** |

In [ ]:
import sys, os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, os.path.join(project_root, 'src'))

from seqcredit_model.config import (
    DATA_DIR, TRANSACTIONS_DIR, USER_FEATURES_FILE, USER_LABELS_FILE
)
from seqcredit_model.credit_model import (
    CreditRiskDataLoader, LogisticRegressionModel, XGBoostModel, ModelEvaluator
)

%matplotlib inline
sns.set_style('whitegrid')
plt.rcParams.update({'figure.dpi': 100, 'font.size': 11, 'axes.titlesize': 12})

STALE_LABELS_FILE = str(DATA_DIR / 'legacy' / 'summary_extended.csv')
CORRECT_LABELS_FILE = str(USER_LABELS_FILE)
FEATURES_FILE = str(USER_FEATURES_FILE)
TRANSACTIONS_DIR = str(TRANSACTIONS_DIR)

print(f'Stale labels:   {STALE_LABELS_FILE}')
print(f'Correct labels: {CORRECT_LABELS_FILE}')

---
## 1. Label File Differences

Before training anything, show concretely how the two label files differ.

In [ ]:
stale = pd.read_csv(STALE_LABELS_FILE)
correct = pd.read_csv(CORRECT_LABELS_FILE)

print('=== STALE labels (summary_extended.csv) ===')
print(f'  Shape: {stale.shape}')
print(f'  credit_risk_label distribution:')
print(stale['credit_risk_label'].value_counts().sort_index().to_string())

print('\n=== CORRECT labels (user_labels.csv) ===')
print(f'  Shape: {correct.shape}')
print(f'  credit_risk_label distribution:')
print(correct['credit_risk_label'].value_counts().sort_index().to_string())

In [ ]:
# Merge on user_id to find per-user differences
merged = stale.merge(correct, on='user_id', suffixes=('_stale', '_correct'))

label_changed = (merged['credit_risk_label_stale'] != merged['credit_risk_label_correct']).sum()
archetype_changed = (merged['credit_archetype_stale'] != merged['credit_archetype_correct']).sum()
txn_count_diff = (merged['total_transactions_stale'] - merged['total_transactions_correct']).abs()

print(f'Total users in both files: {len(merged):,}')
print(f'Users with changed credit_risk_label: {label_changed:,} ({label_changed/len(merged)*100:.1f}%)')
print(f'Users with changed credit_archetype:  {archetype_changed:,} ({archetype_changed/len(merged)*100:.1f}%)')
print(f'Avg absolute txn count difference:    {txn_count_diff.mean():.1f}')
print(f'Max absolute txn count difference:    {txn_count_diff.max():.0f}')

print('\nSample of changed labels (first 5):')
changed = merged[merged['credit_risk_label_stale'] != merged['credit_risk_label_correct']]
print(changed[['user_id', 'credit_risk_label_stale', 'credit_risk_label_correct',
               'total_transactions_stale', 'total_transactions_correct']].head().to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

risk_labels = {-1: 'No Loans', 0: 'Good', 1: 'Late', 2: 'Default'}
risk_colors = {-1: '#95a5a6', 0: '#2ecc71', 1: '#f39c12', 2: '#e74c3c'}

# Label distribution comparison
for ax, (df, title) in zip(axes[:2], [
    (stale, 'Stale Labels (summary_extended.csv)'),
    (correct, 'Correct Labels (user_labels.csv)'),
]):
    vc = df['credit_risk_label'].value_counts().sort_index()
    bars = ax.bar([risk_labels[k] for k in vc.index], vc.values,
                  color=[risk_colors[k] for k in vc.index], edgecolor='white')
    for bar, v in zip(bars, vc.values):
        ax.text(bar.get_x() + bar.get_width()/2, v + 30, f'{v:,}', ha='center', fontsize=9)
    ax.set_title(title)
    ax.set_ylabel('Count')

# Transaction count shift
axes[2].hist(txn_count_diff, bins=40, color='#3498db', edgecolor='white', alpha=0.8)
axes[2].set_xlabel('|stale txn count - correct txn count|')
axes[2].set_ylabel('Number of Users')
axes[2].set_title('Transaction Count Discrepancy per User')
axes[2].axvline(txn_count_diff.mean(), color='red', linestyle='--',
                label=f'Mean diff: {txn_count_diff.mean():.1f}')
axes[2].legend()

plt.suptitle('Label File Comparison: Stale vs Correct', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 2. Train Models on Both Label Sets

Logistic Regression and XGBoost are trained twice — once with stale labels and once with correct labels — using the same features and random seed. The LSTM is excluded to keep runtime reasonable.

In [ ]:
print('=== Training on STALE labels ===')

# Bypass validation so we can intentionally load stale labels
loader_stale = CreditRiskDataLoader(
    features_path=FEATURES_FILE,
    summaries_path=STALE_LABELS_FILE,
    transactions_dir=TRANSACTIONS_DIR,
)
# Skip _validate_data by calling internal steps directly
df_feat = pd.read_csv(FEATURES_FILE)
df_stale = pd.read_csv(STALE_LABELS_FILE)
df_merged = df_feat.merge(df_stale, on='user_id', how='inner')
df_merged = df_merged[df_merged['credit_risk_label'] != -1].copy()
df_merged['default'] = (df_merged['credit_risk_label'] == 2).astype(int)
print(f'  Borrowers: {len(df_merged):,} | Defaults: {df_merged["default"].sum():,} ({df_merged["default"].mean()*100:.1f}%)')

loader_stale._user_ids = df_merged['user_id'].values
drop_cols = ['user_id', 'credit_risk_label', 'credit_archetype', 'default', 'total_transactions_y']
feature_cols = [c for c in df_merged.columns if c not in drop_cols]
X_stale = df_merged[feature_cols].copy()
y_stale = df_merged['default'].copy()
loader_stale._static_data = (X_stale, y_stale)

splits_stale = loader_stale.prepare_static_splits()

lr_stale = LogisticRegressionModel()
lr_stale.fit(splits_stale['X_train_scaled'], splits_stale['y_train'])

xgb_stale = XGBoostModel(scale_pos_weight=loader_stale.get_scale_pos_weight())
xgb_stale.fit(splits_stale['X_train_scaled'], splits_stale['y_train'])

print('  Done.')

In [ ]:
print('=== Training on CORRECT labels ===')

loader_correct = CreditRiskDataLoader(
    features_path=FEATURES_FILE,
    summaries_path=CORRECT_LABELS_FILE,
    transactions_dir=TRANSACTIONS_DIR,
)
splits_correct = loader_correct.prepare_static_splits()

print(f'  Borrowers: {len(splits_correct["y_train"]) + len(splits_correct["y_test"]):,} | '
      f'Defaults: {splits_correct["y_train"].sum() + splits_correct["y_test"].sum():,} '
      f'({(splits_correct["y_train"].sum() + splits_correct["y_test"].sum()) / (len(splits_correct["y_train"]) + len(splits_correct["y_test"]))*100:.1f}%)')

lr_correct = LogisticRegressionModel()
lr_correct.fit(splits_correct['X_train_scaled'], splits_correct['y_train'])

xgb_correct = XGBoostModel(scale_pos_weight=loader_correct.get_scale_pos_weight())
xgb_correct.fit(splits_correct['X_train_scaled'], splits_correct['y_train'])

print('  Done.')

---
## 3. Evaluation

Each model is evaluated on its own held-out test set (same 80/20 stratified split within each label regime).

In [ ]:
eval_stale = ModelEvaluator(splits_stale['y_test'])
eval_stale.add_model('LR (stale)',  lr_stale.predict_proba(splits_stale['X_test_scaled']))
eval_stale.add_model('XGB (stale)', xgb_stale.predict_proba(splits_stale['X_test_scaled']))

eval_correct = ModelEvaluator(splits_correct['y_test'])
eval_correct.add_model('LR (correct)',  lr_correct.predict_proba(splits_correct['X_test_scaled']))
eval_correct.add_model('XGB (correct)', xgb_correct.predict_proba(splits_correct['X_test_scaled']))

stale_table   = eval_stale.get_comparison_table()
correct_table = eval_correct.get_comparison_table()

# Combined table
combined = pd.concat([stale_table, correct_table])
print('=== Model Performance: Stale vs Correct Labels ===')
try:
    display(combined.style.format('{:.4f}').background_gradient(cmap='RdYlGn', axis=0))
except Exception:
    display(combined.round(4))

In [ ]:
# Delta table: correct - stale (positive = improvement from fix)
metrics = ['AUC-ROC', 'AUC-PR', 'F1', 'Precision', 'Recall', 'Accuracy']

delta_rows = []
for model, label in [('LR', 'Logistic Regression'), ('XGB', 'XGBoost')]:
    stale_row   = stale_table.loc[f'{model} (stale)']
    correct_row = correct_table.loc[f'{model} (correct)']
    delta = correct_row - stale_row
    delta.name = label
    delta_rows.append(delta)

delta_df = pd.DataFrame(delta_rows)[metrics]
print('=== Delta (correct - stale): positive = improvement from bug fix ===')
try:
    display(delta_df.style.format('{:+.4f}').background_gradient(cmap='RdYlGn', axis=None))
except Exception:
    display(delta_df.round(4))

---
## 4. Visual Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ROC curves — all four models on the same axes
line_styles = {
    'LR (stale)':   ('#e74c3c', '--'),
    'XGB (stale)':  ('#e67e22', '--'),
    'LR (correct)': ('#2ecc71', '-'),
    'XGB (correct)':('#2980b9', '-'),
}

for name, res in {**eval_stale.results, **eval_correct.results}.items():
    color, ls = line_styles[name]
    axes[0].plot(res['fpr'], res['tpr'], color=color, linestyle=ls, linewidth=2,
                 label=f"{name} (AUC={res['auc_roc']:.3f})")
axes[0].plot([0, 1], [0, 1], 'k:', alpha=0.4, label='Random')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curves: Stale (--) vs Correct (—)')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# PR curves
for name, res in {**eval_stale.results, **eval_correct.results}.items():
    color, ls = line_styles[name]
    axes[1].plot(res['recall_arr'], res['precision_arr'], color=color, linestyle=ls, linewidth=2,
                 label=f"{name} (AP={res['auc_pr']:.3f})")
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('PR Curves: Stale (--) vs Correct (—)')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
metrics_plot = ['AUC-ROC', 'AUC-PR', 'F1', 'Precision', 'Recall']
fig, axes = plt.subplots(1, len(metrics_plot), figsize=(18, 5))

models_plot = ['LR', 'XGB']
stale_colors   = ['#e74c3c', '#e67e22']
correct_colors = ['#2ecc71', '#2980b9']

for ax, metric in zip(axes, metrics_plot):
    x = np.arange(len(models_plot))
    width = 0.35
    stale_vals   = [stale_table.loc[f'{m} (stale)', metric]   for m in models_plot]
    correct_vals = [correct_table.loc[f'{m} (correct)', metric] for m in models_plot]

    bars1 = ax.bar(x - width/2, stale_vals,   width, label='Stale',   color='#e74c3c', alpha=0.8, edgecolor='white')
    bars2 = ax.bar(x + width/2, correct_vals, width, label='Correct', color='#2ecc71', alpha=0.8, edgecolor='white')

    for bar, v in zip(bars1, stale_vals):
        ax.text(bar.get_x() + bar.get_width()/2, v + 0.005, f'{v:.3f}', ha='center', fontsize=8, color='#c0392b')
    for bar, v in zip(bars2, correct_vals):
        ax.text(bar.get_x() + bar.get_width()/2, v + 0.005, f'{v:.3f}', ha='center', fontsize=8, color='#27ae60')

    ax.set_xticks(x)
    ax.set_xticklabels(models_plot)
    ax.set_title(metric)
    ax.set_ylim(0, min(1.15, max(stale_vals + correct_vals) * 1.2))
    ax.legend(fontsize=8)

plt.suptitle('Metric Comparison: Stale vs Correct Labels', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
all_results = {**eval_stale.results, **eval_correct.results}
border_colors = {'stale': '#e74c3c', 'correct': '#2ecc71'}

for ax, (name, res) in zip(axes, all_results.items()):
    cm = res['confusion_matrix']
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Non-Default', 'Default'],
                yticklabels=['Non-Default', 'Default'])
    ax.set_title(name)
    ax.set_ylabel('Actual')
    ax.set_xlabel('Predicted')
    # Colour the border to indicate stale vs correct
    tag = 'stale' if 'stale' in name else 'correct'
    for spine in ax.spines.values():
        spine.set_edgecolor(border_colors[tag])
        spine.set_linewidth(3)

plt.suptitle('Confusion Matrices  |  Red border = stale labels  |  Green border = correct labels',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

---
## 5. Summary

In [ ]:
print('=' * 60)
print('STALE LABELS BUG — IMPACT SUMMARY')
print('=' * 60)

for model in models_plot:
    stale_auc   = stale_table.loc[f'{model} (stale)',   'AUC-ROC']
    correct_auc = correct_table.loc[f'{model} (correct)', 'AUC-ROC']
    stale_pr    = stale_table.loc[f'{model} (stale)',   'AUC-PR']
    correct_pr  = correct_table.loc[f'{model} (correct)', 'AUC-PR']
    delta_auc   = correct_auc - stale_auc
    delta_pr    = correct_pr  - stale_pr
    direction   = 'improved' if delta_auc >= 0 else 'degraded'
    print(f'\n{model}:')
    print(f'  AUC-ROC: {stale_auc:.4f} -> {correct_auc:.4f}  ({delta_auc:+.4f}, {direction})')
    print(f'  AUC-PR:  {stale_pr:.4f}  -> {correct_pr:.4f}   ({delta_pr:+.4f})')

print('\nConclusion:')
print('  The stale labels bug caused models to train on labels that did not')
print('  correspond to the actual transaction files, producing misleading')
print('  evaluation metrics. The above deltas quantify the correction.')